## Imports

In [2]:
import requests
import csv
import os
import re
import io
import calendar
from PyPDF2 import PdfReader
from datetime import datetime

## File Paths, Downloading PDFs, and Helper Functions for Data Extraction

In [13]:


# Base data path using current notebook directory
notebook_dir = os.getcwd()
DATE_PATTERN = re.compile(
    r"^(\d{2})-(\d{2})-(\d{4})(?:_\d+)?$",
    re.IGNORECASE
)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

# Browser-like headers to avoid the Lawrence site blocking Python requests
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,*/*",
    "Referer": "https://www.lawpd.com/",
}

def get_filename_from_content_disposition(header):
    if "filename=" in header.lower():
        return header.split("filename=")[-1].strip('"; ')
    return None

def parse_date_from_filename(filename):
    # Old format: 03-03-2025.pdf
    match = re.search(r"(\d{2})-(\d{2})-(\d{4})", filename)

    if match:
        month, day, year = map(int, match.groups())
        return year, month, day

    # New format:
    # Press_20Log_20-_20Apr_2006__202026_20_2801_29.pdf
    # Press_20Log_20-_20Mar_2027__202026.pdf
    # Press_20Log_20-_20March_2031_202026.pdf

    match = re.search(
        r"Press_20Log_20-_20"
        r"(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
        r"Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
        r"_20(\d{1,2})"
        r"(?:__20|_20)"
        r"(\d{4})",
        filename,
        re.IGNORECASE
    )

    if match:
        month_name, day, year = match.groups()

        month = datetime.strptime(
            month_name[:3].title(), "%b"
        ).month

        return int(year), month, int(day)

    return None
def sanitize_filename(name):
    """
    Remove characters that are illegal or awkward in filenames.
    """
    safe = re.sub(r"[^a-zA-Z0-9_\-. ]+", "_", name)
    return safe[:100]
os.makedirs(DATA_DIR, exist_ok=True) #tracks and lists failures in csv

#_____________________#
## ID INPUTTING HERE ## 
#____________________#



def download_pdfs(start_id=51774, end_id=52774):   ## INPUT THE ID DATES for what you are trying to extract : )
    os.makedirs(DATA_DIR, exist_ok=True)
    failure_csv_path = os.path.join(DATA_DIR, "failures.csv")
    failure_file = open(
    failure_csv_path,
    mode="w",
    newline="",
    encoding="utf-8")
    failure_writer = csv.writer(failure_file)
    failure_writer.writerow(["Document ID", "Reason"])
    
    

    total = end_id - start_id + 1

    skipped = {
        "http": 0,
        "not_pdf": 0,
        "missing_filename": 0,
        "wrong_year": 0,
        "error": 0
    }

    saved = 0

    for count, doc_id in enumerate(range(start_id, end_id + 1), start=1):
        if count % 500 == 0:
            print(f"📦 Progress: {count} / {total} checked")

        url = f"https://www.lawpd.com/DocumentCenter/View/{doc_id}"

        try:
            res = requests.get(
                url,
                headers=HEADERS,
                timeout=20,
                allow_redirects=True
            )

            if res.status_code != 200:
                skipped["http"] += 1
                failure_writer.writerow([doc_id, f"HTTP {res.status_code}"])
                continue

            content_type = res.headers.get("Content-Type", "").lower()
            if "pdf" not in content_type:
                skipped["not_pdf"] += 1
                failure_writer.writerow(
                    [doc_id, f"Not PDF (Content-Type: {content_type})"]
                )
                continue

            filename = get_filename_from_content_disposition(
                res.headers.get("Content-Disposition", "")
            )

            if not filename:
                skipped["missing_filename"] += 1
                failure_writer.writerow([doc_id, "Missing filename"])
                continue

            if not filename.lower().endswith(".pdf"):
                filename += ".pdf"
            filename = re.sub(
                r"_(\d+)(\.pdf)$",
                r"\2",
                filename,
                flags=re.IGNORECASE
            )
            filename = sanitize_filename(filename)

            year, month, day = parse_date_from_filename(filename)

            if year is None:
                skipped["wrong_year"] += 1
                failure_writer.writerow([doc_id, f"Invalid date filename: {filename}"])
                continue

            # Build folder path: .../data/2023_january
            month_folder = f"{year}_{calendar.month_name[month].lower()}"
            save_dir = os.path.join(DATA_DIR, month_folder)
            os.makedirs(save_dir, exist_ok=True)

            new_filename = f"{month:02d}-{day:02d}-{year}.pdf"
            save_path = os.path.join(save_dir, new_filename)

            with open(save_path, "wb") as f:
                f.write(res.content)

            saved += 1
            print(f"[{count}/{total}] ✅ Saved: {new_filename}")

        except Exception as e:
            skipped["error"] += 1
            failure_writer.writerow(
                [doc_id, f"{type(e).__name__}: {e}"]
            )
            if skipped["error"] <= 5:
                print(f"{doc_id}: {type(e).__name__}: {e}")
    failure_file.close()
    print(f"Done. Saved {saved} PDFs to {DATA_DIR}")
    print(f"Skipped: {skipped}")

In [14]:
download_pdfs()

[1/1001] ✅ Saved: 03-26-2026.pdf
[2/1001] ✅ Saved: 04-01-2026.pdf
[3/1001] ✅ Saved: 03-31-2026.pdf
[4/1001] ✅ Saved: 03-31-2026.pdf
[5/1001] ✅ Saved: 03-31-2026.pdf
51780: TypeError: cannot unpack non-iterable NoneType object
[8/1001] ✅ Saved: 03-27-2026.pdf
[9/1001] ✅ Saved: 03-28-2026.pdf
[10/1001] ✅ Saved: 03-29-2026.pdf
[11/1001] ✅ Saved: 03-30-2026.pdf
51785: TypeError: cannot unpack non-iterable NoneType object
[13/1001] ✅ Saved: 04-07-2026.pdf
[15/1001] ✅ Saved: 03-31-2026.pdf
51789: TypeError: cannot unpack non-iterable NoneType object
[17/1001] ✅ Saved: 04-01-2026.pdf
[18/1001] ✅ Saved: 04-07-2026.pdf
[21/1001] ✅ Saved: 04-02-2026.pdf
[22/1001] ✅ Saved: 04-03-2026.pdf
[23/1001] ✅ Saved: 04-03-2026.pdf
[24/1001] ✅ Saved: 04-04-2026.pdf
[25/1001] ✅ Saved: 04-04-2026.pdf
51799: TypeError: cannot unpack non-iterable NoneType object
[27/1001] ✅ Saved: 04-08-2026.pdf
[28/1001] ✅ Saved: 04-09-2026.pdf
[29/1001] ✅ Saved: 04-06-2026.pdf
[30/1001] ✅ Saved: 04-06-2026.pdf
51804: TypeErro

In [6]:
import os

print("Current working directory:", os.getcwd())
print("PDF output directory:", os.path.abspath(DATA_DIR))
print("Output directory exists:", os.path.exists(DATA_DIR))

for root, dirs, files in os.walk(DATA_DIR):
    pdfs = [f for f in files if f.lower().endswith(".pdf")]
    if pdfs:
        print(root, ":", len(pdfs), "PDFs")
        print("Examples:", pdfs[:5])

Current working directory: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts
PDF output directory: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data
Output directory exists: True
